# Week 3: Quantum Kernel Results (N=1000, auto-save)
**Project**: Quantum Kernels on CLIP/BERT Embeddings

**This version saves + downloads everything, including figures:**
- Kernel matrices saved **immediately** after the 83-min computation (the thing lost last time)
- Every figure downloaded as both PNG and PDF
- Results JSON checkpointed

**Inputs:** `embeddings_week2.npz`, `baseline_results.pkl` (from Week 2, genuine N=1000)

**Runtime:** ~85 min (kernel matrix is the bottleneck)

In [ ]:
!pip install pennylane pennylane-lightning --quiet

import pennylane as qml
import numpy as np
import time, json, pickle, os, shutil
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

print(f"PennyLane {qml.__version__}")

---
## Step 0: Storage + checkpoint helpers

Mount Drive (recommended) and define helpers that save + download data files
and figures. Downloads are wrapped so a blocked browser popup never stops the run.

In [ ]:
USE_DRIVE = True   # set False to use downloads only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def _persist_and_download(path):
    """Copy a local file to SAVE_DIR and try to download it."""
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        dst = os.path.join(SAVE_DIR, os.path.basename(path))
        shutil.copy(path, dst)
        print(f"  saved -> {dst}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — file is saved above)")

def checkpoint(filename, save_fn):
    """save_fn(path) writes the file; then we persist + download."""
    save_fn(filename)
    _persist_and_download(filename)

def save_figure(fig, name):
    """Save a matplotlib figure as PNG + PDF, persist + download both."""
    png, pdf = f"{name}.png", f"{name}.pdf"
    fig.savefig(png, bbox_inches='tight', dpi=150)
    fig.savefig(pdf, bbox_inches='tight', dpi=150)
    print(f"Figure '{name}':")
    _persist_and_download(png)
    _persist_and_download(pdf)

print("Helpers ready.")

---
## Step 1: Load Week 2 data

In [ ]:
# Upload embeddings_week2.npz and baseline_results.pkl, OR load from Drive if present
def _find(fname):
    drive_path = os.path.join(SAVE_DIR, fname)
    return drive_path if os.path.exists(drive_path) else fname

data = np.load(_find('embeddings_week2.npz'))
X_tr = data['X_tr_subset'].astype(np.float64)   # (1000, 8)
y_tr = data['y_tr_subset']
X_te = data['X_te_subset'].astype(np.float64)   # (200, 8)
y_te = data['y_te_subset']

with open(_find('baseline_results.pkl'), 'rb') as f:
    baselines = pickle.load(f)

print(f"Train: {X_tr.shape}  Test: {X_te.shape}")
print(f"Range: [{X_tr.min():.4f}, {X_tr.max():.4f}]  (must be in [0, pi])")
print(f"Class balance: train {np.mean(y_tr==1):.1%} pos | test {np.mean(y_te==1):.1%} pos")
assert X_tr.shape[0] == 1000, f"Expected 1000 train, got {X_tr.shape[0]} — re-run Week 2"
print("\nGenuine N=1000 confirmed.")
print("\nClassical baselines:")
for k, v in baselines.items():
    print(f"  {k:<12} acc={v['accuracy']:.4f}  f1={v['f1']:.4f}")

---
## Step 2: IQP quantum kernel

In [ ]:
N_QUBITS = 8
dev = qml.device('lightning.qubit', wires=N_QUBITS)

def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(dev)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(N_QUBITS))
    qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))

quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])

# sanity
assert np.isclose(quantum_kernel(X_tr[0], X_tr[0]), 1.0, atol=1e-5)
t0 = time.time()
for _ in range(5): quantum_kernel(X_tr[0], X_tr[1])
ms = (time.time()-t0)/5*1000
print(f"Sanity OK. Single eval {ms:.1f}ms")
print(f"N=1000 matrix estimate: ~{1000*1000*ms/1000/60:.0f} min")

---
## Step 3: Compute kernel matrix — **saved IMMEDIATELY after**

This is the ~83-minute step. The moment it finishes, the matrices are saved to Drive
and downloaded — before any SVM, before any plot. This is the file you lost last time.

In [ ]:
print(f"Computing {len(X_tr)}x{len(X_tr)} quantum kernel matrix...")
print("Do not close the tab. Saving happens automatically when done.")
t0 = time.time()
K_train = qml.kernels.kernel_matrix(X_tr, X_tr, quantum_kernel)
K_test  = qml.kernels.kernel_matrix(X_te, X_tr, quantum_kernel)
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min")

# ---- CHECKPOINT: kernel matrices, saved before anything else ----
print("\nSaving kernel matrices immediately:")
checkpoint('kernel_matrices_week3.npz',
           lambda p: np.savez(p, K_train_quantum=K_train, K_test_quantum=K_test,
                              X_tr=X_tr, y_tr=y_tr))
print("\nSafe. Even if the session dies now, the expensive work is preserved.")

---
## Step 4: Quantum kernel SVM + comparison

In [ ]:
qsvm   = SVC(kernel='precomputed', C=1.0).fit(K_train, y_tr)
y_pred = qsvm.predict(K_test)
acc_q  = float(accuracy_score(y_te, y_pred))
f1_q   = float(f1_score(y_te, y_pred))

print(f"{'='*50}")
print(f"QUANTUM KERNEL SVM (N={len(X_tr)}, {N_QUBITS} qubits, IQP):")
print(f"  Accuracy : {acc_q:.4f}")
print(f"  F1 score : {f1_q:.4f}")
print(f"{'='*50}\n")
print(f"vs classical baselines (N={len(X_tr)}):")
for name, vals in baselines.items():
    diff = acc_q - vals['accuracy']
    print(f"  vs {name:<12}: {diff:+.4f}  ({diff*100:+.1f}%)")   # fixed sign formatting

---
## Step 5: Assemble results (all same N — no retraining needed)

In [ ]:
results_all = {name: {'accuracy': v['accuracy'], 'f1': v['f1']}
               for name, v in baselines.items()}
results_all['Quantum (IQP)'] = {'accuracy': acc_q, 'f1': f1_q}

print(f"{'Method':<14} {'Accuracy':>10} {'F1':>8}")
print("-"*34)
for name, v in results_all.items():
    print(f"{name:<14} {v['accuracy']:>10.4f} {v['f1']:>8.4f}")

---
## Step 6: Comparison figure (downloaded as PNG + PDF)

In [ ]:
names  = list(results_all.keys())
accs   = [results_all[n]['accuracy'] for n in names]
f1s    = [results_all[n]['f1'] for n in names]
colors = ['#5B8DB8','#5B8DB8','#5B8DB8','#D95F02']

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].bar(names, accs, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
axes[0].set_ylim(0.5, 0.85); axes[0].set_title('Accuracy — N=1000'); axes[0].set_ylabel('Accuracy')
for i, v in enumerate(accs):
    axes[0].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11, fontweight='500')
axes[0].set_xticklabels(names, rotation=15, ha='right')

axes[1].bar(names, f1s, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
axes[1].set_ylim(0.5, 0.85); axes[1].set_title('F1 score — N=1000'); axes[1].set_ylabel('F1')
for i, v in enumerate(f1s):
    axes[1].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11, fontweight='500')
axes[1].set_xticklabels(names, rotation=15, ha='right')

blue   = mpatches.Patch(color='#5B8DB8', label='Classical kernels')
orange = mpatches.Patch(color='#D95F02', label='Quantum kernel (IQP, 8 qubits)')
fig.legend(handles=[blue, orange], loc='upper center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, 1.02))
plt.suptitle('Quantum vs classical SVM kernels on 8-dim BERT embeddings (SST-2)', y=1.06)
plt.tight_layout()
plt.show()

save_figure(fig, 'fig1_comparison')

---
## Step 7: Learning curve (downloaded as PNG + PDF)

In [ ]:
train_sizes = [50, 100, 200, 400, 700, 1000]
lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}
from sklearn.svm import SVC as CSVC

print(f"{'N':>5}  {'Quantum':>9}  {'Linear':>9}  {'RBF':>9}  {'Time':>7}")
print("-"*46)
for n in train_sizes:
    Xn, yn = X_tr[:n], y_tr[:n]
    t0 = time.time()
    # quantum — reuse the big matrix slices where possible
    Kn      = K_train[:n, :n]
    Kn_test = K_test[:, :n]
    accq = accuracy_score(y_te, SVC(kernel='precomputed', C=1.0).fit(Kn, yn).predict(Kn_test))
    lc['Quantum (IQP)'].append(accq)
    for cn, ck in [('Linear','linear'), ('RBF','rbf')]:
        clf = CSVC(kernel=ck, C=1.0, gamma='scale' if ck=='rbf' else 1.0,
                   random_state=42).fit(Xn, yn)
        lc[cn].append(accuracy_score(y_te, clf.predict(X_te)))
    print(f"{n:>5}  {lc['Quantum (IQP)'][-1]:>9.4f}  {lc['Linear'][-1]:>9.4f}  "
          f"{lc['RBF'][-1]:>9.4f}  {time.time()-t0:>6.1f}s")

fig2 = plt.figure(figsize=(8, 4))
cmap = {'Quantum (IQP)':'#D95F02','Linear':'#5B8DB8','RBF':'#2D9C4A'}
for m, vals in lc.items():
    ls = '-' if m == 'Quantum (IQP)' else '--'
    lw = 2.5 if m == 'Quantum (IQP)' else 1.8
    plt.plot(train_sizes, vals, label=m, color=cmap[m], linestyle=ls, linewidth=lw, marker='o', markersize=5)
plt.xlabel('Training set size (N)'); plt.ylabel('Test accuracy')
plt.title('Learning curves — quantum vs classical SVM\n(8-dim BERT embeddings, SST-2)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

save_figure(fig2, 'fig2_learning_curves')

---
## Step 8: Kernel matrix heatmap (downloaded as PNG + PDF)

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
fig3, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = axes[0].imshow(K_train[:100, :100], cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Quantum kernel matrix (first 100×100)')
axes[0].set_xlabel('Sample j'); axes[0].set_ylabel('Sample i')
plt.colorbar(im0, ax=axes[0])
K_rbf = rbf_kernel(X_tr[:100], gamma=1.0/N_QUBITS)
im1 = axes[1].imshow(K_rbf, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('RBF kernel matrix (first 100×100)')
axes[1].set_xlabel('Sample j')
plt.colorbar(im1, ax=axes[1])
plt.suptitle('Kernel geometry — quantum vs RBF (same 100 samples)', y=1.02)
plt.tight_layout()
plt.show()

save_figure(fig3, 'fig3_kernel_heatmap')

---
## Step 9: Save results JSON

In [ ]:
week3_results = {
    'n_qubits': N_QUBITS, 'n_train': int(len(X_tr)), 'n_test': int(len(X_te)),
    'feature_map': 'IQP (Hadamard + RZ + ZZ)',
    'compute_time_min': round(elapsed/60, 1),
    'quantum_accuracy': acc_q, 'quantum_f1': f1_q,
    'all_results': results_all,
    'learning_curve_sizes': train_sizes,
    'learning_curve_results': {k: [float(v) for v in vals] for k, vals in lc.items()},
}
print("Saving results JSON:")
checkpoint('results_week3.json',
           lambda p: json.dump(week3_results, open(p, 'w'), indent=2))

print("\n=== WEEK 3 DONE ===")
print(f"Quantum IQP: acc={acc_q:.4f}  f1={f1_q:.4f}")
print(f"\nAll files in {SAVE_DIR}:")
for f in sorted(os.listdir(SAVE_DIR)):
    print(f"  {f}")

---
## Week 3 complete

### Saved + downloaded (PNG + PDF for every figure):
- `kernel_matrices_week3.npz` — saved immediately after the 83-min compute
- `fig1_comparison` — quantum vs classical bars
- `fig2_learning_curves` — accuracy vs training size
- `fig3_kernel_heatmap` — quantum vs RBF kernel geometry
- `results_week3.json` — all numbers

### Note on the learning curve:
This version slices the big kernel matrix (`K_train[:n,:n]`) instead of recomputing —
so the learning curve is nearly instant now, not another 40 minutes.

### Week 4:
Load `kernel_matrices_week3.npz` for the IQP-1L reuse, then run the ablations.